# Chapter 01. 베스트셀러 데이터 이해와 기본 전처리

이번 Notebook에서는 교보문고 베스트셀러 Excel 데이터를 이용해 **텍스트 분석 전에 필요한 기본 전처리**를 진행합니다.

진행 순서는 다음과 같습니다.

**Excel 불러오기 → 데이터 구조 확인 → 필요한 컬럼 선택 → 컬럼명 정리 → 결측치 처리 → 판매가 숫자 변환 → 발행일 날짜 변환 → 문자열 공백 정리 → 중복 확인 → 최종 검증 → CSV 저장**

각 실습은 **AI에게 질문 → AI 답변 → 코드 실행 → 결과 확인 및 정리** 순서로 작성했습니다.

## 실습 1. Excel 파일 불러오기

### AI에게 질문

> Python과 pandas를 처음 배우고 있습니다.  
> 현재 교보문고 베스트셀러 Excel 파일이 있습니다.  
> pandas를 이용해 이 파일을 `df`라는 DataFrame으로 불러오고, 앞의 5행을 확인하는 가장 간단한 코드를 작성해 주세요.  
> 초보자가 이해하기 쉽도록 불필요하게 복잡한 코드는 사용하지 말고, 각 코드가 무엇을 하는지 짧게 설명해 주세요.

### AI 답변

먼저 `pandas`를 불러온 뒤 `pd.read_excel()`로 Excel 파일을 읽으면 됩니다.  
읽어 온 표는 `df`라는 변수에 저장하고, `type(df)`와 `df.head()`를 사용하면 정상적으로 불러왔는지 확인할 수 있습니다.

In [ ]:
# pandas 라이브러리를 pd라는 이름으로 불러옵니다.
import pandas as pd

# 읽어 올 Excel 파일의 위치를 적습니다.
# 현재 VS Code의 작업 위치가 프로젝트 루트이므로 아래 경로를 사용합니다.
file_path = "notebooks/book-text-ml/교보문고_종합_베스트셀러_상품리스트.xlsx"

# Excel 파일을 읽어서 df라는 DataFrame에 저장합니다.
df = pd.read_excel(file_path)

# df가 pandas DataFrame인지 확인합니다.
print(type(df))

# 데이터의 앞 5행을 확인합니다.
df.head()

### 실습 1 결과 확인 및 정리

- `pd.read_excel()`은 Excel 파일을 pandas의 **DataFrame** 형태로 읽어 오는 함수입니다.
- DataFrame은 Excel처럼 **행과 열로 이루어진 표**라고 생각하면 됩니다.
- `type(df)` 결과가 `pandas.core.frame.DataFrame` 형태로 나오면 DataFrame으로 잘 읽힌 것입니다.
- `df.head()`는 기본적으로 앞의 5행을 보여 줍니다.
- 코드가 오류 없이 실행된 것만 확인하지 않고, 실제로 **교보문고 도서 데이터가 보이는지** 확인해야 합니다.

## 실습 2. 데이터의 크기와 컬럼 확인하기

### AI에게 질문

> pandas DataFrame `df`를 불러온 상태입니다.  
> 초보자가 데이터의 기본 구조를 확인하려고 합니다.  
> 다음 내용을 각각 확인할 수 있는 간단한 pandas 코드를 작성해 주세요.  
> 1. 전체 행과 컬럼 개수  
> 2. 컬럼 이름 목록  
> 3. 각 컬럼의 자료형  
> 4. 각 컬럼의 결측치 개수  
> 5. 앞의 5행  
> `shape`, `columns`, `dtypes`, `isna()`, `head()` 정도의 기본 기능을 사용해 주세요.

### AI 답변

전처리를 시작하기 전에 데이터의 현재 상태를 먼저 확인하는 것이 좋습니다.  
`shape`는 데이터 크기, `columns`는 컬럼 이름, `dtypes`는 자료형, `isna().sum()`은 결측치 개수를 확인할 때 사용합니다.

In [ ]:
# 전체 데이터의 크기를 확인합니다.
# 결과는 (행 개수, 컬럼 개수) 형태입니다.
print("데이터 크기:", df.shape)

# 컬럼 이름을 확인합니다.
print("\n컬럼 이름:")
print(df.columns)

# 각 컬럼의 자료형을 확인합니다.
print("\n자료형:")
print(df.dtypes)

# 각 컬럼에 비어 있는 값이 몇 개인지 확인합니다.
print("\n결측치 개수:")
print(df.isna().sum())

# 실제 데이터 앞 5행도 다시 확인합니다.
df.head()

### 실습 2 결과 확인 및 정리

전처리 전에 데이터 구조를 먼저 확인했습니다.

- `df.shape` → 전체 **행 수와 컬럼 수**
- `df.columns` → 현재 DataFrame에 들어 있는 **컬럼 이름**
- `df.dtypes` → 각 컬럼이 정수, 문자열 등 어떤 **자료형**인지 확인
- `df.isna().sum()` → 컬럼마다 **결측치가 몇 개 있는지** 확인
- `df.head()` → 실제 값이 어떤 형태인지 확인

중요한 점은 **숫자처럼 보이는 값도 실제로는 문자열일 수 있고**, 비어 있는 값도 있을 수 있다는 것입니다. 따라서 데이터를 수정하기 전에 먼저 상태를 확인해야 합니다.

## 실습 3. 분석에 필요한 컬럼만 선택하기

### AI에게 질문

> pandas DataFrame `df`에 교보문고 베스트셀러 데이터가 있습니다.  
> 다음 컬럼만 선택해서 새로운 DataFrame `df_books`를 만들고 싶습니다.  
> - 순위
> - 판매상품 ID
> - 상품명
> - 판매가
> - 인물
> - 출판사
> - 발행(출시)일자
> - 분야
>
> Python 초보자가 이해할 수 있도록 가장 간단한 pandas 코드로 작성해 주세요.  
> 컬럼이 실제로 잘 선택되었는지 확인하는 코드도 함께 작성해 주세요.

### AI 답변

사용할 컬럼 이름을 리스트로 만든 뒤 `df[리스트]` 형태로 선택하면 됩니다.  
원본 `df`를 그대로 남겨 두기 위해 `.copy()`를 사용해서 `df_books`를 따로 만들겠습니다.

In [ ]:
# 이번 분석에서 사용할 컬럼 이름을 리스트로 만듭니다.
selected_columns = [
    "순위",
    "판매상품 ID",
    "상품명",
    "판매가",
    "인물",
    "출판사",
    "발행(출시)일자",
    "분야",
]

# df에서 필요한 컬럼만 선택해서 새로운 DataFrame을 만듭니다.
# copy()를 사용하면 원본 df와 별도로 안전하게 작업할 수 있습니다.
df_books = df[selected_columns].copy()

# 선택한 컬럼 이름과 데이터 앞 5행을 확인합니다.
print(df_books.columns)
df_books.head()

### 실습 3 결과 확인 및 정리

원본 Excel에는 여러 컬럼이 있지만 이번 분석에서 모두 사용할 필요는 없습니다.

이번에는 이후의 도서 텍스트 분석과 기본 정보 확인에 필요한 컬럼만 골라 `df_books`를 만들었습니다.

특히 `.copy()`를 사용하면 이후 `df_books`의 값을 수정하더라도 원본 `df`를 그대로 남겨 둘 수 있습니다.  
이렇게 **원본 데이터와 작업용 데이터를 분리**해 두면 실수했을 때 다시 원본에서 시작하기 쉽습니다.

## 실습 4. 컬럼 이름을 사용하기 쉽게 정리하기

### AI에게 질문

> DataFrame `df_books`의 컬럼 이름을 일부 변경하려고 합니다.  
> - 판매상품 ID → 판매상품ID
> - 인물 → 저자
> - 발행(출시)일자 → 발행일
>
> pandas `rename()`을 이용해 가장 간단하게 변경하는 코드를 작성해 주세요.  
> 변경 후 `columns`를 출력해서 결과를 확인하는 코드도 작성해 주세요.

### AI 답변

`rename(columns={기존 이름: 새 이름})` 형태로 원하는 컬럼 이름만 바꿀 수 있습니다.  
변경 후에는 `df_books.columns`를 출력해서 제대로 바뀌었는지 확인하면 됩니다.

In [ ]:
# rename()으로 길거나 사용하기 불편한 컬럼 이름을 바꿉니다.
df_books = df_books.rename(columns={
    "판매상품 ID": "판매상품ID",
    "인물": "저자",
    "발행(출시)일자": "발행일",
})

# 변경된 컬럼 이름을 확인합니다.
print(df_books.columns)

### 실습 4 결과 확인 및 정리

컬럼 이름을 바꾼 이유는 **이후 코드에서 반복해서 사용하기 편하게 만들기 위해서**입니다.

예를 들어 `발행(출시)일자`보다 `발행일`이 훨씬 짧고 읽기 쉽습니다.  
`rename()`은 데이터 값 자체를 바꾸는 것이 아니라 **컬럼의 이름만 변경**합니다.

## 실습 5. 결측치 확인하고 처리하기

### AI에게 질문 1

> DataFrame `df_books`의 컬럼별 결측치 개수를 확인하고 싶습니다.  
> pandas `isna()`와 `sum()`을 이용해 가장 간단한 코드를 작성해 주세요.  
> 그리고 결측치가 있는 컬럼만 보고 싶을 때 사용할 수 있는 쉬운 코드도 알려 주세요.

### AI 답변 1

`df_books.isna().sum()`을 실행하면 모든 컬럼의 결측치 개수를 확인할 수 있습니다.  
그 결과에서 0보다 큰 값만 필터링하면 결측치가 있는 컬럼만 볼 수 있습니다.

In [ ]:
# 모든 컬럼의 결측치 개수를 확인합니다.
missing_count = df_books.isna().sum()
print(missing_count)

# 결측치가 1개 이상 있는 컬럼만 확인합니다.
print("\n결측치가 있는 컬럼:")
print(missing_count[missing_count > 0])

### AI에게 질문 2

> DataFrame `df_books`에서 저자 컬럼의 결측치는 '미상', 분야 컬럼의 결측치는 '미분류'로 바꾸고 싶습니다.  
> pandas `fillna()`를 이용해서 초보자가 이해하기 쉬운 코드로 작성해 주세요.  
> 처리 후 결측치 개수를 다시 확인하는 코드도 작성해 주세요.

### AI 답변 2

먼저 처리 전 결측치 개수를 저장한 뒤 `fillna()`를 이용해 값을 채우면 됩니다.  
처리 후에는 다시 `isna().sum()`을 실행해서 실제로 결측치가 처리되었는지 확인해야 합니다.

In [ ]:
# 나중에 결과를 비교하기 위해 처리 전 결측치 개수를 저장합니다.
author_missing_before = df_books["저자"].isna().sum()
category_missing_before = df_books["분야"].isna().sum()

# 저자 결측치는 '미상'으로 채웁니다.
df_books["저자"] = df_books["저자"].fillna("미상")

# 분야 결측치는 '미분류'로 채웁니다.
df_books["분야"] = df_books["분야"].fillna("미분류")

# 처리 후 결측치 개수를 다시 확인합니다.
print(df_books.isna().sum())

### 실습 5 결과 확인 및 정리

**결측치**는 값이 비어 있는 데이터를 의미합니다.

이번 실습에서 중요한 점은 결측치를 발견했다고 해서 바로 행을 삭제하지 않았다는 것입니다.

- 저자가 비어 있어도 상품명 분석에는 사용할 수 있습니다.
- 분야가 비어 있어도 다른 정보는 정상일 수 있습니다.
- 따라서 이번 기준에서는 저자는 `미상`, 분야는 `미분류`로 채웠습니다.

전처리에서는 **처리하기 전 확인 → 처리 → 다시 확인** 순서가 중요합니다.

## 실습 6. 판매가를 숫자로 바꾸기

### AI에게 질문

> DataFrame `df_books`의 판매가 컬럼에 '16,200', '17,820'처럼 쉼표가 포함된 가격 문자열이 있습니다.  
> 1. 쉼표를 제거하고
> 2. 숫자형으로 변환하고
> 3. 변환 후 dtype을 확인하고 싶습니다.
>
> pandas를 사용해 초보자가 이해하기 쉬운 코드로 작성해 주세요.

### AI 답변

판매가를 문자열 형태로 확인한 뒤 `str.replace()`로 쉼표를 제거하고, `pd.to_numeric()`으로 실제 숫자형으로 바꾸면 됩니다.  
변환 후에는 `dtype`과 `describe()`를 확인해서 숫자로 정상 변환되었는지 점검하겠습니다.

In [ ]:
# 변환 전 판매가의 자료형을 확인합니다.
print("변환 전 자료형:", df_books["판매가"].dtype)

# pandas의 string 자료형으로 바꾼 뒤 쉼표를 제거합니다.
# 예: "16,200" -> "16200"
df_books["판매가"] = df_books["판매가"].astype("string")
df_books["판매가"] = df_books["판매가"].str.replace(",", "", regex=False)

# 문자열 판매가를 실제 숫자로 변환합니다.
# 숫자로 바꿀 수 없는 값은 NaN으로 처리합니다.
df_books["판매가"] = pd.to_numeric(df_books["판매가"], errors="coerce")

# 변환 결과를 확인합니다.
print("변환 후 자료형:", df_books["판매가"].dtype)
print(df_books["판매가"].head())

# 평균, 최소값, 최대값 등 기본 통계도 확인합니다.
df_books["판매가"].describe()

### 실습 6 결과 확인 및 정리

사람 눈에는 `16,200`이 숫자로 보이지만 pandas에서는 쉼표 때문에 문자열로 읽힐 수 있습니다.

문자열 상태에서는 평균이나 합계를 계산하기 어렵기 때문에 다음 순서로 처리했습니다.

1. 문자열 형태로 변환
2. 쉼표 제거
3. `pd.to_numeric()`으로 숫자형 변환
4. `dtype`과 실제 값을 다시 확인
5. `describe()`로 값의 범위도 확인

여기서 `errors="coerce"`는 숫자로 바꿀 수 없는 값이 있다면 오류를 멈추는 대신 `NaN`으로 처리하라는 의미입니다.

## 실습 7. 발행일을 날짜형으로 바꾸기

### AI에게 질문

> DataFrame `df_books`의 발행일 컬럼이 20260701처럼 YYYYMMDD 형식으로 저장되어 있습니다.  
> pandas `to_datetime()`을 이용해 날짜 자료형으로 변환하고 싶습니다.  
> `format`을 명시해서 변환하는 가장 쉬운 코드를 작성해 주세요.  
> 변환 후 dtype과 앞의 5개 값을 확인하는 코드도 작성해 주세요.

### AI 답변

먼저 발행일을 문자열 형태로 정리한 뒤 `pd.to_datetime()`에 `format="%Y%m%d"`를 지정하면 됩니다.  
Excel에서 숫자로 읽히면서 뒤에 `.0`이 붙는 경우도 처리한 뒤 날짜로 변환하겠습니다.

In [ ]:
# 발행일을 pandas의 string 자료형으로 바꿉니다.
df_books["발행일"] = df_books["발행일"].astype("string")

# Excel에서 20260701.0처럼 읽힌 경우 맨 뒤의 .0만 제거합니다.
df_books["발행일"] = df_books["발행일"].str.replace(r"\.0$", "", regex=True)

# YYYYMMDD 형식을 실제 날짜형으로 변환합니다.
df_books["발행일"] = pd.to_datetime(
    df_books["발행일"],
    format="%Y%m%d",
    errors="coerce"
)

# 변환된 값과 자료형을 확인합니다.
print(df_books["발행일"].head())
print("자료형:", df_books["발행일"].dtype)

### 실습 7 결과 확인 및 정리

`20260701`은 연도·월·일 정보를 가지고 있지만 그대로는 일반 숫자나 문자열처럼 다뤄질 수 있습니다.

`pd.to_datetime()`을 사용해 날짜형으로 바꾸면 이후에 연도, 월, 요일 등을 쉽게 분석할 수 있습니다.

`format="%Y%m%d"`의 의미는 다음과 같습니다.

- `%Y` → 연도 4자리
- `%m` → 월 2자리
- `%d` → 일 2자리

변환 뒤에는 반드시 앞의 값과 `dtype`을 확인해서 날짜가 정상적으로 만들어졌는지 확인해야 합니다.

## 실습 8. 문자열 앞뒤 공백 정리하기

### AI에게 질문

> DataFrame `df_books`에서 상품명, 저자, 출판사, 분야 컬럼의 앞뒤 공백을 제거하고 싶습니다.  
> pandas 문자열 함수 `str.strip()`을 이용해 초보자가 이해하기 쉬운 반복문 코드로 작성해 주세요.  
> 처리 후 각 컬럼의 앞의 몇 개 값을 확인하는 방법도 알려 주세요.

### AI 답변

공백을 정리할 컬럼 이름을 리스트로 만들고 `for` 반복문을 사용하면 같은 코드를 여러 번 쓰지 않아도 됩니다.  
각 컬럼을 pandas의 `string` 자료형으로 바꾼 뒤 `str.strip()`을 적용하겠습니다.

In [ ]:
# 공백을 정리할 문자열 컬럼을 리스트로 만듭니다.
string_columns = ["상품명", "저자", "출판사", "분야"]

# 리스트에서 컬럼 이름을 하나씩 꺼내 앞뒤 공백을 제거합니다.
for column in string_columns:
    # astype("string")은 결측치를 문자열 "nan"으로 바꾸지 않고 유지할 수 있습니다.
    df_books[column] = df_books[column].astype("string").str.strip()

# 정리한 문자열 컬럼의 앞 5행을 확인합니다.
df_books[string_columns].head()

### 실습 8 결과 확인 및 정리

문자열의 앞뒤에 공백이 있으면 사람 눈에는 같은 값처럼 보여도 Python에서는 서로 다른 문자열로 처리됩니다.

예를 들어 `"소설"`과 `"소설 "`은 서로 다른 값입니다.

`str.strip()`은 문자열의 **앞과 뒤에 있는 불필요한 공백**을 제거합니다.  
여러 컬럼에 같은 작업을 해야 하므로 `for` 반복문을 사용했습니다.

이번 단계에서는 특수문자 제거, 형태소 분석 같은 복잡한 텍스트 전처리는 하지 않고 **기본적인 공백 정리만** 진행했습니다.

## 실습 9. 중복 데이터 확인하기

### AI에게 질문

> DataFrame `df_books`에서 중복 데이터를 확인하고 싶습니다.  
> 1. 전체 행이 완전히 같은 중복 행 개수
> 2. 판매상품ID가 중복된 행 개수
> 3. 상품명이 중복된 행 개수
>
> 를 각각 pandas `duplicated()`로 확인하는 간단한 코드를 작성해 주세요.  
> 중복을 바로 삭제하지 말고 개수만 확인하도록 작성해 주세요.

### AI 답변

`duplicated().sum()`을 사용하면 중복으로 판단된 행의 개수를 셀 수 있습니다.  
이번 단계에서는 전체 행, 판매상품ID, 상품명을 각각 기준으로 확인만 하고 삭제하지 않겠습니다.

In [ ]:
# 전체 행이 완전히 같은 중복 개수를 셉니다.
row_duplicate_count = df_books.duplicated().sum()

# 판매상품ID가 중복된 개수를 셉니다.
product_id_duplicate_count = df_books["판매상품ID"].duplicated().sum()

# 상품명이 중복된 개수를 셉니다.
title_duplicate_count = df_books["상품명"].duplicated().sum()

# 결과를 확인합니다.
print("전체 행 완전 중복:", row_duplicate_count)
print("판매상품ID 중복:", product_id_duplicate_count)
print("상품명 중복:", title_duplicate_count)

### 실습 9 결과 확인 및 정리

중복을 확인할 때는 **무엇을 기준으로 중복이라고 볼 것인지** 생각해야 합니다.

- 전체 행이 모두 같으면 완전 중복일 가능성이 높습니다.
- 판매상품ID는 상품을 구분하는 식별자이므로 중복 여부를 확인할 가치가 있습니다.
- 하지만 상품명은 같은 제목의 다른 판본이나 개정판이 있을 수 있으므로 제목이 같다고 바로 삭제하면 안 됩니다.

따라서 이번 단계에서는 중복 행을 삭제하지 않고 **개수만 확인**했습니다.

## 실습 10. 전처리 결과 최종 검증하기

### AI에게 질문

> 전처리가 끝난 pandas DataFrame `df_books`를 최종 검증하고 싶습니다.  
> 다음 내용을 한 번에 확인하는 초보자용 코드를 작성해 주세요.
> - shape
> - columns
> - dtypes
> - isna().sum()
> - 판매가 앞의 5개 값
> - 발행일 앞의 5개 값
> - 판매상품ID 중복 개수
> - head()
>
> 결과를 자동으로 좋다/나쁘다 판단하지 말고, 확인할 수 있도록 print 중심으로 간단하게 작성해 주세요.

### AI 답변

전처리 후에는 처음에 확인했던 데이터 구조를 다시 확인해야 합니다.  
코드가 실행됐다는 사실만 보는 것이 아니라, 실제 값과 자료형이 원하는 형태가 되었는지 직접 읽어 보는 것이 중요합니다.

In [ ]:
# 전체 행과 컬럼 개수
print("데이터 크기:", df_books.shape)

# 컬럼 이름
print("\n컬럼 이름:")
print(df_books.columns)

# 각 컬럼의 자료형
print("\n자료형:")
print(df_books.dtypes)

# 남아 있는 결측치 개수
print("\n결측치:")
print(df_books.isna().sum())

# 판매가가 숫자로 잘 변환되었는지 확인
print("\n판매가 앞의 5개:")
print(df_books["판매가"].head())

# 발행일이 날짜로 잘 변환되었는지 확인
print("\n발행일 앞의 5개:")
print(df_books["발행일"].head())

# 판매상품ID 중복 개수
print("\n판매상품ID 중복 개수:")
print(df_books["판매상품ID"].duplicated().sum())

# 최종 데이터 앞 5행
df_books.head()

### 실습 10 결과 확인 및 정리

전처리는 코드를 실행했다고 끝나는 것이 아닙니다. 마지막에 실제 결과를 다시 확인해야 합니다.

특히 다음을 확인합니다.

- 필요한 컬럼만 남았는가?
- 판매가가 숫자 자료형으로 바뀌었는가?
- 발행일이 날짜 자료형으로 바뀌었는가?
- 결측치를 처리하려던 컬럼이 의도대로 처리되었는가?
- 데이터 행이 예상하지 않게 사라지지는 않았는가?
- 중복 개수가 어느 정도인지 확인했는가?

예를 들어 코드 오류가 없어도 날짜가 전부 `NaT`로 바뀌었다면 전처리가 제대로 된 것이 아닙니다.  
따라서 **실행 성공과 분석 성공은 다르다**는 점을 기억해야 합니다.

## 실습 11. 전처리 데이터 저장하기

### AI에게 질문

> 전처리가 끝난 pandas DataFrame `df_books`를 `book_bestseller_clean.csv` 파일로 저장하고 싶습니다.  
> Windows와 Excel에서도 한글이 잘 보이도록 `utf-8-sig` 인코딩을 사용하고, DataFrame의 인덱스는 저장하지 않도록 해 주세요.  
> pandas `to_csv()`를 이용해 간단한 코드로 작성해 주세요.

### AI 답변

`to_csv()`를 사용하면 DataFrame을 CSV 파일로 저장할 수 있습니다.  
`index=False`로 불필요한 인덱스 저장을 막고, `encoding="utf-8-sig"`를 사용해 Excel에서 한글이 깨질 가능성을 줄이겠습니다.

In [ ]:
# 저장할 CSV 파일의 위치를 지정합니다.
output_path = "notebooks/book-text-ml/book_bestseller_clean.csv"

# 전처리한 df_books를 CSV 파일로 저장합니다.
# index=False : 왼쪽의 0, 1, 2 같은 인덱스 번호는 저장하지 않습니다.
# encoding="utf-8-sig" : Excel에서 한글이 깨질 가능성을 줄입니다.
df_books.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)

### 실습 11 결과 확인 및 정리

전처리가 끝난 데이터를 `book_bestseller_clean.csv`로 저장했습니다.

이 파일은 원본 Excel 파일을 그대로 복사한 것이 아니라, 지금까지 다음 작업을 거친 **전처리 완료 데이터**입니다.

- 필요한 컬럼 선택
- 컬럼 이름 정리
- 저자/분야 결측치 처리
- 판매가 숫자형 변환
- 발행일 날짜형 변환
- 문자열 공백 정리
- 중복 여부 확인

다음 Chapter에서는 이 CSV 파일을 다시 불러와 상품명 단어 빈도와 텍스트 분석을 진행할 수 있습니다.

## 실습 12. 실제 실행 결과를 바탕으로 Markdown 작성하기

### AI에게 질문

> 교보문고 베스트셀러 데이터 기본 전처리를 완료했습니다.  
> Notebook에서 실제로 확인한 원본 데이터 크기, 선택한 컬럼 수, 저자 결측치, 분야 결측치, 판매상품ID 중복, 판매가 자료형, 발행일 자료형을 이용해 짧은 Markdown 설명을 작성하고 싶습니다.  
> 다음 세 부분으로 작성해 주세요.
>
> 1. 무엇을 확인했는지  
> 2. 어떤 전처리를 했는지  
> 3. 다음 분석을 위해 어떤 상태가 되었는지  
>
> 실제로 확인하지 않은 숫자는 추측하지 않도록 해 주세요.

### AI 답변

숫자를 임의로 적지 않고, 앞에서 실제로 계산한 변수 값을 문장 안에 넣도록 하겠습니다.  
아래 셀은 `f-string`을 사용해 실제 실행 결과를 Markdown 문장에 자동으로 넣습니다.

In [ ]:
# Markdown 형태로 결과를 보여 주기 위해 필요한 기능을 불러옵니다.
from IPython.display import display, Markdown

# f-string을 사용하면 { } 안에 실제 변수 값을 넣을 수 있습니다.
summary_md = f"""
### Chapter 01 전처리 결과

#### 1. 무엇을 확인했는지
원본 데이터의 크기, 컬럼 이름, 자료형, 결측치와 중복 여부를 확인했습니다.
원본 데이터는 **{df.shape[0]}행 × {df.shape[1]}열**이며,
이번 분석에서는 **{df_books.shape[1]}개 컬럼**을 선택했습니다.

#### 2. 어떤 전처리를 했는지
저자 결측치 **{author_missing_before}개**는 '미상'으로,
분야 결측치 **{category_missing_before}개**는 '미분류'로 처리했습니다.
판매가는 숫자 자료형으로 변환했고, 발행일은 날짜 자료형으로 변환했습니다.
문자열 컬럼의 앞뒤 공백도 정리했습니다.
판매상품ID 중복 개수는 **{product_id_duplicate_count}개**입니다.

#### 3. 다음 분석을 위한 상태
전처리가 끝난 데이터를 **book_bestseller_clean.csv**로 저장했습니다.
이제 다음 Chapter에서 상품명을 이용한 단어 빈도와 텍스트 분석에 사용할 수 있습니다.
"""

# 위에서 만든 Markdown 문장을 Notebook 화면에 출력합니다.
display(Markdown(summary_md))

### 실습 12 결과 확인 및 정리

이 단계에서는 **실제 Notebook에서 계산한 값만 사용해 결과를 정리**했습니다.

직접 숫자를 입력해서 추측한 것이 아니라 `df.shape`, 결측치 변수, 중복 개수 변수 등을 문장 안에 넣었기 때문에, 셀을 처음부터 정상적으로 실행하면 실제 결과가 Markdown에 반영됩니다.

여기서 `f-string`은 문자열 앞에 `f`를 붙이고 `{변수}`를 적으면 변수의 실제 값이 문장 안에 들어가는 기능입니다.

## 이번 Chapter에서 꼭 기억할 것

이번 실습에서 중요한 것은 복잡한 코드를 많이 작성하는 것이 아니라 **데이터를 확인하고 필요한 처리를 한 뒤 다시 확인하는 과정**입니다.

- 결측치를 발견했다고 바로 삭제하지 않습니다.
- 중복을 발견했다고 바로 삭제하지 않습니다.
- 숫자처럼 보여도 실제 자료형을 확인합니다.
- 날짜처럼 보여도 실제 날짜 자료형인지 확인합니다.
- AI가 만든 코드가 오류 없이 실행되었다고 해서 분석이 맞다는 뜻은 아닙니다.

이번 Chapter의 핵심 흐름은 다음과 같습니다.

**확인 → 판단 → 전처리 → 다시 확인**

## 확인 문제

### 질문 1. 왜 모든 원본 컬럼을 사용하지 않고 필요한 컬럼만 선택했나요?

분석에 필요하지 않은 컬럼까지 가지고 있으면 데이터가 복잡해지고 이후 코드를 작성하기도 어려워질 수 있습니다.  
이번 분석에서 실제로 사용할 정보만 선택하면 데이터의 목적이 분명해지고 전처리 과정도 단순해집니다.

### 질문 2. 판매가가 문자열이라면 어떤 문제가 발생할 수 있나요?

문자열은 숫자가 아니기 때문에 평균, 합계, 최대값 같은 숫자 계산을 정상적으로 하기 어렵습니다.  
따라서 판매가에서 쉼표를 제거하고 숫자 자료형으로 변환해야 합니다.

### 질문 3. 결측치를 확인한 뒤 바로 모든 행을 삭제하면 안 되는 이유는 무엇인가요?

한 컬럼의 값이 비어 있어도 다른 컬럼의 정보는 분석에 사용할 수 있기 때문입니다.  
예를 들어 저자가 비어 있어도 상품명은 단어 분석에 사용할 수 있습니다.  
따라서 분석 목적을 확인한 뒤 삭제할지, 다른 값으로 채울지 결정해야 합니다.

### 질문 4. 같은 상품명이 두 번 등장했다고 해서 바로 중복 데이터라고 판단하기 어려운 이유는 무엇인가요?

같은 제목의 책이라도 개정판, 다른 판본, 세트 상품처럼 실제로는 서로 다른 상품일 수 있습니다.  
따라서 상품명만 같다는 이유로 행을 바로 삭제하면 정상 데이터까지 지울 수 있습니다.

### 질문 5. AI가 작성한 코드가 오류 없이 실행되었다면 그것만으로 분석이 끝난 것일까요?

아닙니다. 코드가 실행되었다는 것은 문법적인 오류가 없다는 뜻일 뿐입니다.  
결과값이 실제 데이터와 맞는지, 자료형이 원하는 형태인지, 값이 잘못 사라지지는 않았는지 직접 확인해야 분석이 끝납니다.

## 제출 전 확인

Notebook을 제출하기 전에 처음부터 마지막 셀까지 다시 실행하면서 다음 내용을 확인합니다.

- [ ] Excel 파일이 정상적으로 로드된다.
- [ ] 데이터 크기와 컬럼을 확인했다.
- [ ] 필요한 컬럼만 선택했다.
- [ ] 컬럼 이름을 정리했다.
- [ ] 결측치를 확인하고 처리했다.
- [ ] 판매가를 숫자형으로 변환했다.
- [ ] 발행일을 날짜형으로 변환했다.
- [ ] 문자열 앞뒤 공백을 정리했다.
- [ ] 중복 데이터를 확인했다.
- [ ] 최종 데이터를 다시 검증했다.
- [ ] `book_bestseller_clean.csv`를 저장했다.
- [ ] 실제 실행 결과를 이용한 Markdown이 표시된다.
- [ ] Notebook 전체를 다시 실행했을 때 오류가 없다.